In [155]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [156]:
df = pd.read_csv('vehicle_dataset_v4A.csv')

In [157]:
df.head()

,category,make,model,model_year,gear,fuel_type,engine_cc,mileage_km,price
0,pickup,Mahindra,Bolero,2015,Manual,Diesel,2500,42315,2075000
1,car,Suzuki,800,2011,Manual,Petrol,800,73142,1725000
2,car,Toyota,Belta,2012,Automatic,Petrol,1300,63002,3350000
3,car,Tata,Indica V2,2004,Manual,Petrol,1400,90000,1200000
4,car,Suzuki,800,2006,Manual,Petrol,800,77000,1055000


In [158]:
df.shape

(14193, 9)

In [159]:
# seperating the features as X and target as y
# dropping the price column and take all the remaining columns as the features as X
# price column taking for Y axis
X = df.drop('price', axis=1)
y = df['price']

In [160]:
from sklearn.model_selection import train_test_split

In [161]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25)

In [162]:
X_train.shape

(10644, 8)

In [163]:
X_train.head()

,category,make,model,model_year,gear,fuel_type,engine_cc,mileage_km
2040,van,Hyundai,H100,2000,Manual,Diesel,2600,214420
12580,car,Chery,QQ,2005,Manual,Petrol,800,68000
7807,van,Toyota,Dolphin,2003,Automatic,Petrol,2000,156450
3754,car,Mazda,3,2007,Automatic,Petrol,1500,98000
504,car,Toyota,Allion,2001,Automatic,Petrol,1500,121000


In [164]:
# determine categorical and numerical features
# because we need to provide the column transformer with seperate numerical and categorical data.
numerical_ix = X_train.select_dtypes(include=['int64', 'float64']).columns
categorical_ix = X_train.select_dtypes(include=['object', 'bool']).columns

In [165]:
categorical_ix

Index(['category', 'make', 'model', 'gear', 'fuel_type'], dtype='object')

In [166]:
numerical_ix

Index(['model_year', 'engine_cc', 'mileage_km'], dtype='object')

In [167]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score, GridSearchCV

In [168]:
# define the data preparation for the columns
# One hot encoding is transffering the categorical column into the numerical columns in the memory,
# StandardScaler() -Using numerical features column with scaling to convert the large value of data into smaller size so that the model can easily understand 
ct = ColumnTransformer([
    ('ohe',OneHotEncoder(), categorical_ix), 
    ('ss',StandardScaler(), numerical_ix)
])

In [169]:
ct.fit(X_train)

ColumnTransformer(transformers=[('ohe', OneHotEncoder(),
                                 Index(['category', 'make', 'model', 'gear', 'fuel_type'], dtype='object')),
                                ('ss', StandardScaler(),
                                 Index(['model_year', 'engine_cc', 'mileage_km'], dtype='object'))])

In [170]:
# create the pipeline with both column transformer and model.
# first the pipeline will perform the elements in the column transformer one by one and then it will run the model
# basically implementing the pipeline in the order in which it should perform
pipe = Pipeline([
    ('ct',ct),
    ('model',RandomForestRegressor(n_jobs=-1))
])

In [171]:
# train the pipeline with the training data.
# The pipeline learns the structure and get ready to make predictions when new data is presented.
pipe.fit(X_train, y_train)

Pipeline(steps=[('ct',
                 ColumnTransformer(transformers=[('ohe', OneHotEncoder(),
                                                  Index(['category', 'make', 'model', 'gear', 'fuel_type'], dtype='object')),
                                                 ('ss', StandardScaler(),
                                                  Index(['model_year', 'engine_cc', 'mileage_km'], dtype='object'))])),
                ('model', RandomForestRegressor(n_jobs=-1))])

In [172]:
# making predictions using pipeline
prediction = pipe.predict(X_test)

In [173]:
# Import matrics to calculate the performance
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [174]:
# checking the r2 score [accuracy]
r2_score(y_test, prediction)

0.8654803103465911

In [175]:
print('MAE:', mean_absolute_error(y_test, prediction))
print('MSE:', mean_squared_error(y_test, prediction))
print('RMSE:', np.sqrt(mean_squared_error(y_test, prediction)))

MAE: 362147.64578219445
MSE: 919755571653.1082
RMSE: 959038.8791144539


# Grid Search

In [176]:
params = {
    'model__n_estimators': [340,350,360],
    'model__min_samples_split': [10,12,15],
    'model__min_samples_leaf': [1,2,3]
}

In [177]:
grid_search = GridSearchCV(estimator=pipe,param_grid=params, n_jobs=-1,verbose=3)

In [178]:
X_train.head(1)

,category,make,model,model_year,gear,fuel_type,engine_cc,mileage_km
2040,van,Hyundai,H100,2000,Manual,Diesel,2600,214420


In [179]:
grid_search.fit(X_train,y_train)

Fitting 5 folds for each of 27 candidates, totalling 135 fits


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 4 concurrent workers.
[Parallel(n_jobs=-1)]: Done  24 tasks      | elapsed:  6.7min
[Parallel(n_jobs=-1)]: Done 120 tasks      | elapsed: 30.5min
[Parallel(n_jobs=-1)]: Done 135 out of 135 | elapsed: 33.6min finished


GridSearchCV(estimator=Pipeline(steps=[('ct',
                                        ColumnTransformer(transformers=[('ohe',
                                                                         OneHotEncoder(),
                                                                         Index(['category', 'make', 'model', 'gear', 'fuel_type'], dtype='object')),
                                                                        ('ss',
                                                                         StandardScaler(),
                                                                         Index(['model_year', 'engine_cc', 'mileage_km'], dtype='object'))])),
                                       ('model',
                                        RandomForestRegressor(n_jobs=-1))]),
             n_jobs=-1,
             param_grid={'model__min_samples_leaf': [1, 2, 3],
                         'model__min_samples_split': [10, 12, 15],
                         'model__n_estimat

In [180]:
grid_search.best_params_

{'model__min_samples_leaf': 1,
 'model__min_samples_split': 10,
 'model__n_estimators': 340}

In [181]:
predictions = grid_search.predict(X_test)

In [182]:
# checking the r2 score [accuracy]
r2_score(y_test, predictions)

0.8705739612992311

In [183]:
print('MAE:', mean_absolute_error(y_test, predictions))
print('MSE:', mean_squared_error(y_test, predictions))
print('RMSE:', np.sqrt(mean_squared_error(y_test, predictions)))

MAE: 352643.2105184648
MSE: 884928596837.6182
RMSE: 940706.4349932014


In [184]:
import pickle

# Save the Module to file
Pkl_Filename = "Pickle_RF_Model2.pkl"  
with open(Pkl_Filename, 'wb') as file:  
    pickle.dump(grid_search, file)